In [2]:
import os
import tarfile
import random
import re
import math
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import _download_asset
from torch.utils.data import Dataset, DataLoader
from jiwer import wer
from tqdm.auto import tqdm
import whisper

c:\Users\itism\Desktop\test\ml-asr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [4]:
class Config:
    sr = 16000
    batch_size = 32
    epochs = 325
    lr = 1e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"

    n_fft = 512
    hop_length = 128
    win_length = 512

    gamma = 1.5
    sigma_min = 0.05
    sigma_max = 0.5
    t_eps = 0.03

    alpha = 0.5
    beta = 1.0

    tar_path = "../data/raw/ru_train_0.tar"
    tsv_path = "../data/raw/train(1).tsv"
    extract_dir = "../data/raw/ru_train_data"

    duration_sec = 3.0
    target_samples = int(sr * duration_sec)

In [5]:
class SDE:
    def __init__(self, config):
        self.gamma = config.gamma
        self.sigma_min = config.sigma_min
        self.sigma_max = config.sigma_max
        self.log_sig = math.log(self.sigma_max / self.sigma_min)

    def g(self, t):
        return self.sigma_min * (self.sigma_max / self.sigma_min)**t * math.sqrt(2 * self.log_sig)

    def marginal_prob(self, x0, y, t):
        t = t.view(-1, 1, 1, 1)
        mean = torch.exp(-self.gamma * t) * x0 + (1 - torch.exp(-self.gamma * t)) * y
        
        factor = 2 * self.gamma + 2 * self.log_sig
        var = (self.sigma_min**2 / factor) * (
            (self.sigma_max / self.sigma_min)**(2*t) - torch.exp(-2 * self.gamma * t)
        )
        return mean, torch.sqrt(var)

def compress_stft(c, alpha=0.5):
    mag = torch.abs(c)
    phase = torch.angle(c)
    return (mag ** alpha) * torch.exp(1j * phase)

def decompress_stft(c_tilde, alpha=0.5):
    mag = torch.abs(c_tilde)
    phase = torch.angle(c_tilde)
    return (mag ** (1.0 / alpha)) * torch.exp(1j * phase)

In [6]:
class ComplexConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1):
        super().__init__()
        self.conv_r = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, dilation)
        self.conv_i = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, dilation)

    def forward(self, x):
        xr, xi = x.real, x.imag
        real_out = self.conv_r(xr) - self.conv_i(xi)
        imag_out = self.conv_r(xi) + self.conv_i(xr)
        return torch.complex(real_out, imag_out)

class ComplexConvTranspose2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, output_padding=0):
        super().__init__()
        self.conv_r = nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, output_padding)
        self.conv_i = nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, output_padding)

    def forward(self, x):
        xr, xi = x.real, x.imag
        real_out = self.conv_r(xr) - self.conv_i(xi)
        imag_out = self.conv_r(xi) + self.conv_i(xr)
        return torch.complex(real_out, imag_out)

class ComplexLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.fc_r = nn.Linear(in_features, out_features)
        self.fc_i = nn.Linear(in_features, out_features)

    def forward(self, x):
        xr, xi = x.real, x.imag
        real_out = self.fc_r(xr) - self.fc_i(xi)
        imag_out = self.fc_r(xi) + self.fc_i(xr)
        return torch.complex(real_out, imag_out)

class ComplexReLU(nn.Module):
    def forward(self, x):
        return torch.complex(F.relu(x.real), F.relu(x.imag))

class ComplexBatchNorm2d(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.bn_r = nn.BatchNorm2d(num_features)
        self.bn_i = nn.BatchNorm2d(num_features)

    def forward(self, x):
        return torch.complex(self.bn_r(x.real), self.bn_i(x.imag))

class RealGaussianFourierProjection(nn.Module):
    def __init__(self, embed_dim=128, scale=30):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim // 2) * scale, requires_grad=False)

    def forward(self, t):
        x = t[:, None] * self.W[None, :] * 2 * math.pi
        return torch.cat([torch.cos(x), torch.sin(x)], dim=-1)

In [7]:
class StandardBlock(nn.Module):
    def __init__(self, in_c, out_c, kernel_size, stride, is_decoder=False, output_padding=0):
        super().__init__()
        padding = (kernel_size[0] // 2, kernel_size[1] // 2)

        if is_decoder:
            self.conv = nn.ConvTranspose2d(in_c, out_c, kernel_size, stride, padding=padding, output_padding=output_padding)
        else:
            self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride, padding=padding)

        self.norm = nn.BatchNorm2d(out_c)
        self.act = nn.ReLU()

        self.time_embed = nn.Sequential(
            nn.Linear(128, out_c),
            nn.ReLU()
        )

    def forward(self, x, t_emb):
        h = self.conv(x)
        h = self.norm(h)
        
        t_embed = self.time_embed(t_emb)
        t_embed = t_embed.view(t_embed.shape[0], t_embed.shape[1], 1, 1)
        
        h = h + t_embed
        return self.act(h)

class ScoreModelDCUNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.t_proj = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU()
        )

        self.enc1 = nn.Sequential(nn.Conv2d(4, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU())
        self.down1 = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.down2 = nn.MaxPool2d(2)

        self.bottleneck = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())

        self.up2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.dec2 = nn.Sequential(nn.Conv2d(64 + 32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())

        self.up1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.dec1 = nn.Sequential(nn.Conv2d(32 + 16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU())

        self.final = nn.Conv2d(16, 2, 3, padding=1)

    def forward(self, x_t, t, y):
        t_emb = self.t_proj(t.unsqueeze(-1)).view(-1, 64, 1, 1)

        x = torch.cat([x_t.real, x_t.imag, y.real, y.imag], dim=1)

        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))

        b = self.bottleneck(self.down2(e2))
        b = b + t_emb 

        def pad_match(target, source):
            diff_f = source.shape[2] - target.shape[2]
            diff_t = source.shape[3] - target.shape[3]
            return F.pad(target, [diff_t // 2, diff_t - diff_t // 2, diff_f // 2, diff_f - diff_f // 2])

        d2 = self.up2(b)
        d2 = pad_match(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = pad_match(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        out = self.final(d1)
        out = pad_match(out, x_t)

        return torch.complex(out[:, 0:1, :, :], out[:, 1:2, :, :])

In [8]:
def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

def get_snr_scale(signal, noise, snr_db):
    sig_power = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noise_power = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    return torch.sqrt(target_noise_power / (noise_power + 1e-8))

def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)

    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]

    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

if not os.path.exists(Config.extract_dir):
    os.makedirs(Config.extract_dir, exist_ok=True)
    with tarfile.open(Config.tar_path, "r") as tar: 
        tar.extractall(path=Config.extract_dir)

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

import soundfile as sf
import torch

babble_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
global BABBLE_WAVEFORM

waveform_np, sr_b = sf.read(babble_path, always_2d=True)
BABBLE_WAVEFORM = torch.tensor(waveform_np.T, dtype=torch.float32)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
global RIR_WAVEFORM

waveform_r_np, sr_r = sf.read(rir_path, always_2d=True)
RIR_WAVEFORM = torch.tensor(waveform_r_np.T, dtype=torch.float32)

RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

In [9]:
class WaveformDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        file_list = []
        for r, _, fs in os.walk(data_dir):
            for f in fs:
                if f.endswith('.mp3') and f in ref_dict:
                    file_list.append(os.path.join(r, f))
        self.files = sorted(file_list)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        wav_np, sr = librosa.load(file_path, sr=None, mono=False)
        wav = torch.from_numpy(wav_np)
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)

        if sr != Config.sr:
            wav = T.Resample(sr, Config.sr)(wav)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if wav.shape[-1] > Config.target_samples:
            start = random.randint(0, wav.shape[-1] - Config.target_samples) if self.is_train else 0
            wav = wav[:, start:start + Config.target_samples]
        else:
            wav = F.pad(wav, (0, Config.target_samples - wav.shape[-1]))

        clean = wav
        noisy = apply_noise(clean) if self.is_train else clean
        return noisy.squeeze(0), clean.squeeze(0), self.ref_dict[os.path.basename(file_path)]

def exact_collate_fn(batch):
    noisy, clean, texts = zip(*batch)
    return torch.stack(noisy), torch.stack(clean), list(texts)

In [10]:
def sgmse_loss(model, sde, clean_stft, noisy_stft, eps=1e-5):
    B = clean_stft.shape[0]

    t = torch.rand(B, device=clean_stft.device) * (1. - eps) + eps
    mean, std = sde.marginal_prob(clean_stft, noisy_stft, t)
    z = (torch.randn_like(clean_stft) + 1j * torch.randn_like(clean_stft)) / math.sqrt(2)

    x_t = mean + std * z
    score_pred = model(x_t, t, noisy_stft)

    loss = torch.mean(torch.abs(score_pred * std + z)**2)
    return loss

In [11]:
dataset = WaveformDataset(Config.extract_dir, reference_dict, is_train=True)
generator = torch.Generator().manual_seed(42)
train_size = int(0.9 * len(dataset))
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size], generator=generator)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True, collate_fn=exact_collate_fn, num_workers=4, pin_memory=True, persistent_workers=True)

config = Config()
model = ScoreModelDCUNet().to(config.device)
sde = SDE(config)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
window = torch.hann_window(config.n_fft).to(config.device)

In [ ]:
for epoch in range(1, config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for noisy, clean, _ in pbar:
        noisy, clean = noisy.to(config.device), clean.to(config.device)
        optimizer.zero_grad()

        X = torch.stft(noisy, config.n_fft, config.hop_length, window=window, return_complex=True, center=True)
        S = torch.stft(clean, config.n_fft, config.hop_length, window=window, return_complex=True, center=True)

        X_c = compress_stft(X, alpha=config.alpha).unsqueeze(1)
        S_c = compress_stft(S, alpha=config.alpha).unsqueeze(1)

        loss = sgmse_loss(model, sde, S_c, X_c, eps=config.t_eps)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"SGMSE Loss": f"{loss.item():.4f}"})

torch.save(model.state_dict(), "../models/sgmse_weights.pth")

Epoch 1:   0%|          | 0/745 [00:00<?, ?it/s]c:\Users\itism\Desktop\test\ml-asr\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [13]:
model.load_state_dict(torch.load('../models/sgmse_weights.pth', map_location='cpu'))

seed_everything(42)

In [14]:
def evaluate_sgmse(model, sde, config, device, val_dataset, limit=20, N_steps=30):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    window = torch.hann_window(config.n_fft).to(device)

    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": []} for n in noise_types}

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="WER Eval"):

            _, clean_wav, ref_text = val_dataset[idx]
            ref_text = clean_text(ref_text)

            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav.unsqueeze(0), force_type=n_type, file_seed=idx).to(device)

                noisy_wav_1d = noisy_wav.squeeze()

                Y = torch.stft(noisy_wav_1d, config.n_fft, config.hop_length, window=window, return_complex=True, center=True)
                Y_c = compress_stft(Y, alpha=config.alpha).unsqueeze(0).unsqueeze(0)

                z = (torch.randn_like(Y_c) + 1j * torch.randn_like(Y_c)) / math.sqrt(2)
                x = Y_c + sde.sigma_max * z

                t_steps = torch.linspace(1.0, config.t_eps, N_steps, device=device)
                dt = (1.0 - config.t_eps) / N_steps

                for i in range(N_steps):
                    t_val = t_steps[i].expand(1)
                    score = model(x, t_val, Y_c)

                    gamma = sde.gamma
                    g = sde.g(t_val).view(-1, 1, 1, 1)

                    drift = gamma * (Y_c - x) - (g**2) * score

                    z_step = (torch.randn_like(x) + 1j * torch.randn_like(x)) / math.sqrt(2)
                    diffusion = g * math.sqrt(dt) * z_step if i < N_steps - 1 else 0

                    x = x - drift * dt + diffusion

                S_hat_c = x.squeeze()
                S_hat = decompress_stft(S_hat_c, alpha=config.alpha)

                denoised_wav = torch.istft(S_hat, config.n_fft, config.hop_length, window=window, center=True)

                if denoised_wav.shape[-1] > noisy_wav_1d.shape[-1]:
                    denoised_wav = denoised_wav[..., :noisy_wav_1d.shape[-1]]
                elif denoised_wav.shape[-1] < noisy_wav_1d.shape[-1]:
                    denoised_wav = F.pad(denoised_wav, (0, noisy_wav_1d.shape[-1] - denoised_wav.shape[-1]))

                t_n = asr.transcribe(noisy_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_wav.cpu().numpy(), fp16=False, language='ru')['text']

                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))

    print(f"\n{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denoised':<10}")
    for n in noise_types:
        wn, wd = np.mean(stats[n]["wer_n"]), np.mean(stats[n]["wer_d"])
        print(f"{n:<10} | {wn:<10.4f} | {wd:<10.4f}")

evaluate_sgmse(model, sde, config, config.device, val_ds, limit=20, N_steps=30)

c:\Users\itism\Desktop\test\ml-asr\.venv\Lib\site-packages\whisper\__init__.py:69: UserWarning: C:\Users\itism\.cache\whisper\large-v3.pt exists, but the SHA256 checksum does not match; re-downloading the file
  warnings.warn(
  3%|█                                    | 85.9M/2.88G [00:07<04:22, 11.4MiB/s]


KeyboardInterrupt: 

In [ ]:
from jiwer import process_words
import numpy as np
import torch
import math
import torch.nn.functional as F
from tqdm.auto import tqdm
import whisper

def evaluate_sgmse_components(model, sde, config, device, val_dataset, limit=None, N_steps=30):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    window = torch.hann_window(config.n_fft).to(device)

    noise_types = ['babble', 'rir', 'white']
    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            ref_text = clean_text(ref_text)

            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav.unsqueeze(0), force_type=n_type, file_seed=idx).to(device)
                noisy_wav_1d = noisy_wav.squeeze()

                Y = torch.stft(noisy_wav_1d, config.n_fft, config.hop_length, window=window, return_complex=True, center=True)
                Y_c = compress_stft(Y, alpha=config.alpha).unsqueeze(0).unsqueeze(0)

                z = (torch.randn_like(Y_c) + 1j * torch.randn_like(Y_c)) / math.sqrt(2)
                x = Y_c + sde.sigma_max * z

                t_steps = torch.linspace(1.0, config.t_eps, N_steps, device=device)
                dt = (1.0 - config.t_eps) / N_steps

                for i in range(N_steps):
                    t_val = t_steps[i].expand(1)
                    score = model(x, t_val, Y_c)

                    gamma = sde.gamma
                    g = sde.g(t_val).view(-1, 1, 1, 1)

                    drift = gamma * (Y_c - x) - (g**2) * score

                    z_step = (torch.randn_like(x) + 1j * torch.randn_like(x)) / math.sqrt(2)
                    diffusion = g * math.sqrt(dt) * z_step if i < N_steps - 1 else 0

                    x = x - drift * dt + diffusion

                S_hat_c = x.squeeze()
                S_hat = decompress_stft(S_hat_c, alpha=config.alpha)

                denoised_wav = torch.istft(S_hat, config.n_fft, config.hop_length, window=window, center=True)

                if denoised_wav.shape[-1] > noisy_wav_1d.shape[-1]:
                    denoised_wav = denoised_wav[..., :noisy_wav_1d.shape[-1]]
                elif denoised_wav.shape[-1] < noisy_wav_1d.shape[-1]:
                    denoised_wav = F.pad(denoised_wav, (0, noisy_wav_1d.shape[-1] - denoised_wav.shape[-1]))

                t_n = asr.transcribe(noisy_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_wav.cpu().numpy(), fp16=False, language='ru')['text']

                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)

                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits

                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    stats[n_type]["wer_n"].append(0.0)
                    stats[n_type]["s_n"].append(0.0)
                    stats[n_type]["d_n"].append(0.0)
                    stats[n_type]["i_n"].append(0.0)

                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits

                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    stats[n_type]["wer_d"].append(0.0)
                    stats[n_type]["s_d"].append(0.0)
                    stats[n_type]["d_d"].append(0.0)
                    stats[n_type]["i_d"].append(0.0)

    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print(header)
    print("-" * 65)

    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])

        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])

        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"

        print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")

seed_everything(42)
evaluate_sgmse_components(model, sde, config, config.device, val_ds, limit=20, N_steps=30)

Evaluation (Macro-average):   0%|          | 0/20 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Noise    | WER_N   (S/D/I) | WER_D   (S/D/I) | Gain WER
-----------------------------------------------------------------
babble   | 0.5376 (0.1392/0.3875/0.0108) | 0.9690 (0.2874/0.6421/0.0396) | -0.4315 
rir      | 1.0000 (0.3116/0.6884/0.0000) | 1.0000 (0.2780/0.7220/0.0000) | 0.0000  
white    | 0.5892 (0.2199/0.3631/0.0063) | 0.9728 (0.3980/0.5498/0.0250) | -0.3836 
